# Spike 03 — English OCR with Chandra 2

**Goal:** Extract structured English text from the Tranquil City English page images using Chandra 2.

**Time box:** 1-2 hours

**Question to answer:** Does Chandra produce well-structured Markdown (especially tables) from English report pages?

**Done when:** `.md` files in `ocr_output/english/` contain readable text with tables preserved as Markdown.

---

### Why Chandra 2 for English?

| Model | Overall Average (90 langs) | Arabic |
|---|---|---|
| **Chandra 2** | **72.7%** ✅ | 68.4% |
| Gemini 2.5 Flash | 60.8% | 84.4% |

Chandra wins on overall quality and outputs structured Markdown that preserves tables, column headers, and multi-column layouts.

---

### ⚠️ `chandra-ocr` is a local model package — no cloud API exists

The package runs the model on your machine. Two options depending on your hardware:

| Option | Hardware needed | Speed | Setup |
|---|---|---|---|
| **A — Google Colab** | Free T4 GPU (cloud) | ~5 sec/page | Upload images, run, download output |
| **B — CPU mode** | 16GB+ RAM | ~2-5 min/page | Run locally, go make coffee |
| **C — Gemini fallback** | None | ~4 sec/page | Already working from Spike 02 |

**Run Option A or C for the spike. Option B only if you have time and patience.**

## Option A — Google Colab (Recommended)

Run this in a **Colab notebook** with GPU runtime (`Runtime → Change runtime type → T4 GPU`):

```python
# In Colab — run as-is, no changes needed
!pip install chandra-ocr pillow -q

from PIL import Image
from chandra.model.hf import load_hf_model, generate_hf
from chandra.model.schema import BatchInputItem
from pathlib import Path
import torch

print("CUDA Available:", torch.cuda.is_available())  # should print True on Colab

model = load_hf_model("datalab-to/chandra-ocr-2", torch_dtype="float16")

OUTPUT_DIR = Path("ocr_output_english")
OUTPUT_DIR.mkdir(exist_ok=True)

# Upload your images_en/ folder to Colab Files first
image_files = sorted(Path("images_en").glob("*.png"))

for img_path in image_files:
    out_path = OUTPUT_DIR / img_path.with_suffix(".md").name
    if out_path.exists():
        continue

    image = Image.open(img_path).convert("RGB")
    image.thumbnail((1600, 1600))

    batch  = [BatchInputItem(image=image, prompt_type="ocr_layout")]
    result = generate_hf(batch, model)[0]

    out_path.write_text(result.markdown, encoding="utf-8")
    print(f"✅ {img_path.name} → {len(result.markdown)} chars")

# Then download the ocr_output_english/ folder back to your machine
# and copy .md files into: C:\Dev\mda-urban-intelligence\ocr_output\english\
```

**Steps:**
1. Go to colab.research.google.com → New notebook
2. Change runtime to T4 GPU
3. Upload your `data/images_en/` folder (just the first 14 pages)
4. Paste and run the code above
5. Download `ocr_output_english/` and copy into `ocr_output/english/`

In [ ]:
## Option B — CPU Mode (Local, Slow)

⚠️ **Check your RAM first.** The model needs ~14-28GB RAM to load.  
If you have less, the process will crash or thrash. Use Option A or C instead.

```python
import psutil
ram_gb = psutil.virtual_memory().total / (1024**3)
print(f"Your RAM: {ram_gb:.1f} GB")
print("Recommendation:", "✅ Try CPU mode" if ram_gb >= 14 else "❌ Not enough RAM — use Colab or Gemini")
```

## Option B — CPU Mode Code

If your RAM check passed (≥14GB), run this modified version of the original code.  
Key changes from the GPU version:

| Original (GPU) | Modified (CPU) |
|---|---|
| `torch_dtype="float16"` | `torch_dtype=torch.float32` — float16 is GPU-only |
| No device specified | `device_map="cpu"` — explicit CPU |
| No memory flag | `low_cpu_mem_usage=True` — loads weights progressively |
| `thumbnail((1600,1600))` | `thumbnail((800,800))` — smaller = faster on CPU |

In [ ]:
from PIL import Image
from chandra.model.hf import load_hf_model, generate_hf
from chandra.model.schema import BatchInputItem
from pathlib import Path
import torch
import time

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}  ← expected False on CPU-only machine")
print()
print("Loading model on CPU — this will take 2-5 minutes and use significant RAM...")
print("Watch your Task Manager RAM usage. If it hits 95%+ → kill the kernel and use Colab.")

model = load_hf_model(
    "datalab-to/chandra-ocr-2",
    torch_dtype      = torch.float32,   # ← float32 for CPU stability (not float16)
    device_map       = "cpu",           # ← explicit CPU
    low_cpu_mem_usage= True             # ← loads shard-by-shard, reduces peak RAM
)

print("\n✅ Model loaded on CPU")

## Option B — OCR Function & Bulk Run

In [ ]:
def ocr_page_chandra_cpu(image_path: str) -> str:
    """
    Run Chandra OCR on a single page image using CPU.
    Expect 2-5 minutes per page on a typical low-end machine.
    """
    image = Image.open(image_path).convert("RGB")
    image.thumbnail((800, 800))   # smaller than GPU version — faster on CPU

    batch  = [BatchInputItem(image=image, prompt_type="ocr_layout")]
    result = generate_hf(batch, model)[0]
    return result.markdown


OUTPUT_DIR = Path("../ocr_output/english")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

en_images = sorted(Path("../data/images_en").glob("*.png"))[:14]  # same 14 pages as Arabic
print(f"Processing {len(en_images)} pages on CPU...\n")

for i, img_path in enumerate(en_images):
    out_path = OUTPUT_DIR / img_path.with_suffix(".md").name

    if out_path.exists():
        print(f"⏭  Skipping {img_path.name} — already done")
        continue

    print(f"[{i+1}/{len(en_images)}] {img_path.name}...", end=" ", flush=True)
    t0 = time.time()

    try:
        text = ocr_page_chandra_cpu(str(img_path))
        out_path.write_text(text, encoding="utf-8")
        elapsed = time.time() - t0
        print(f"✅ {len(text)} chars — {elapsed:.0f}s")
    except Exception as e:
        print(f"❌ {e}")

print(f"\nDone.")

---
## Option C — Gemini Fallback (Fastest, Unblocks Pipeline Now)

**Use this if:** Colab is unavailable and CPU mode is too slow or crashes.

Gemini is already proven from Spike 02. The only change is the prompt — English-only, no Arabic instructions.  
Quality trade-off: Gemini overall = 60.8% vs Chandra 72.7%, but good enough to validate Spike 04 and 05.  
You can always re-run Spike 03 properly with Chandra later on Colab.

In [ ]:
from google import genai
from google.genai import types
from dotenv import load_dotenv
from pathlib import Path
import os, time

load_dotenv(dotenv_path="../.env")
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))
MODEL  = "gemini-2.5-flash"

EN_OCR_PROMPT = """\
You are a specialized OCR engine. Extract ALL text from this document image.
Rules:
- Extract all English text accurately
- Preserve table structure using Markdown table format (| col1 | col2 |)
- Preserve heading hierarchy (# for main titles, ## for sections)
- Preserve numbers, percentages, and statistics exactly as shown
- Preserve any Arabic text you encounter as-is
- Output ONLY the extracted text — no commentary, no explanation
"""

def ocr_page_gemini(image_path: str) -> str:
    with open(image_path, "rb") as f:
        image_bytes = f.read()
    response = client.models.generate_content(
        model    = MODEL,
        contents = [
            types.Part.from_text(text=EN_OCR_PROMPT),
            types.Part.from_bytes(data=image_bytes, mime_type="image/png")
        ]
    )
    return response.text


OUTPUT_DIR = Path("../ocr_output/english")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

en_images = sorted(Path("../data/images_en").glob("*.png"))[:14]
print(f"Processing {len(en_images)} English pages via Gemini...\n")

for i, img_path in enumerate(en_images):
    out_path = OUTPUT_DIR / img_path.with_suffix(".md").name

    if out_path.exists():
        print(f"⏭  Skipping {img_path.name} — already done")
        continue

    print(f"[{i+1}/{len(en_images)}] {img_path.name}...", end=" ", flush=True)
    try:
        text = ocr_page_gemini(str(img_path))
        out_path.write_text(text, encoding="utf-8")
        print(f"✅ {len(text)} chars")
    except Exception as e:
        print(f"❌ {e}")

    if i < len(en_images) - 1:
        time.sleep(4)

print("\n✅ Done")

---
## Step — Side-by-Side Quality Check

Whichever option you ran above, verify the output here before moving to Spike 04.

In [ ]:
from IPython.display import display, Image as IPImage

OUTPUT_DIR = Path("../ocr_output/english")
en_images  = sorted(Path("../data/images_en").glob("*.png"))
md_files   = sorted(OUTPUT_DIR.glob("*.md"))

PAGE_TO_CHECK = 6   # page 6 — likely has real content, not a cover

if len(md_files) >= PAGE_TO_CHECK and len(en_images) >= PAGE_TO_CHECK:
    img_file  = en_images[PAGE_TO_CHECK - 1]
    text_file = md_files[PAGE_TO_CHECK - 1]

    print(f"=== ORIGINAL IMAGE: {img_file.name} ===")
    display(IPImage(filename=str(img_file), width=650))

    print(f"\n=== EXTRACTED TEXT: {text_file.name} ===")
    content = text_file.read_text(encoding="utf-8")
    print(content[:2000])
    print(f"\n[{len(content)} total chars]")
else:
    print(f"Only {len(md_files)} pages processed — adjust PAGE_TO_CHECK")

## Spike Summary

In [ ]:
OUTPUT_DIR = Path("../ocr_output/english")
en_images  = sorted(Path("../data/images_en").glob("*.png"))[:14]
md_files   = sorted(OUTPUT_DIR.glob("*.md"))

total_chars  = sum(f.read_text(encoding="utf-8").__len__() for f in md_files)
tables_found = sum(1 for f in md_files if "|" in f.read_text(encoding="utf-8"))

print("=" * 50)
print("SPIKE 03 — RESULTS")
print("=" * 50)
print(f"Pages processed      : {len(md_files)} / {len(en_images)}")
print(f"Total chars extracted: {total_chars:,}")
print(f"Avg chars per page   : {total_chars // max(len(md_files), 1):,}")
print(f"Pages with tables    : {tables_found}")
print()

if len(md_files) == len(en_images) and total_chars > 1000:
    print("✅ SPIKE PASSED — English OCR output ready for Spike 04 (PageIndex)")
    print(f"   Files in: {OUTPUT_DIR.resolve()}")
    print()
    print("Note: if you used Option C (Gemini), mark this as 'passed with fallback'")
    print("      Proper Chandra validation can be done on Colab later.")
else:
    print(f"⚠️  SPIKE INCOMPLETE — {len(en_images) - len(md_files)} pages still missing")

## Step 7 — Spike Summary

In [ ]:
md_files = sorted(OUTPUT_DIR.glob("*.md"))
total_chars = sum(f.read_text(encoding="utf-8").__len__() for f in md_files)

print("=" * 50)
print("SPIKE 03 — RESULTS")
print("=" * 50)
print(f"Pages processed     : {len(md_files)} / {len(en_images)}")
print(f"Total chars extracted: {total_chars:,}")
print(f"Avg chars per page  : {total_chars // max(len(md_files),1):,}")
print()

# Check table preservation
tables_found = sum(1 for f in md_files if '|' in f.read_text(encoding='utf-8'))
print(f"Pages with Markdown tables: {tables_found}")
print()

if len(md_files) == len(en_images) and total_chars > 1000:
    print("✅ SPIKE PASSED — English OCR output ready for Spike 04 (PageIndex)")
    print(f"   Files in: {OUTPUT_DIR.resolve()}")
else:
    print("⚠️  SPIKE INCOMPLETE")